# Alternative Uncertainty Quantification for MinHash: Binomial and Bayesian Methods

This notebook demonstrates practical alternatives to EVT-MinHash for uncertainty quantification in MinHash-based near-duplicate detection.

## What This Demo Does

1. **Implements two alternative approaches:**
   - **Binomial Confidence Intervals**: Clopper-Pearson (exact) and Wilson score intervals
   - **Bayesian Approach**: Beta prior informed by document length

2. **Compares against bootstrap baseline** on short text datasets

3. **Evaluates**: Coverage probability, interval width, computational cost

## Dataset
This demo uses a curated subset of short text documents from tweet_eval (sentiment).

In [ ]:
# Install dependencies - follows aii-colab pattern for Colab compatibilityimport subprocess, sysdef _pip(*a):    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])# loguru is NOT pre-installed on Colab (always install)_pip('loguru')# Core packages (pre-installed on Colab, install locally to match Colab env)if 'google.colab' not in sys.modules:    _pip('numpy==2.0.2', 'scipy==1.16.3')print("Dependencies installed successfully!")

In [ ]:
# Imports - from original method.py with additional imports for visualizationfrom loguru import loggerfrom pathlib import Pathimport jsonimport numpy as npimport hashlibimport timeimport gcfrom collections import defaultdictfrom scipy import statsfrom scipy.stats import beta, binomimport sysimport matplotlib.pyplot as plt# Setup loggerlogger.remove()logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")print("All imports successful!")

In [ ]:
# Data loading helper - uses GitHub URL with local fallbackGITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-1ba2b3-why-extreme-value-theory-fails-for-minha/main/round-2/experiment-2/demo/mini_demo_data.json"def load_data():    """Load data from GitHub URL with local fallback."""    try:        import urllib.request        with urllib.request.urlopen(GITHUB_DATA_URL) as response:            return json.loads(response.read().decode())    except Exception as e:        print(f"GitHub URL failed: {e}")    if os.path.exists("mini_demo_data.json"):        with open("mini_demo_data.json") as f:            return json.load(f)    raise FileNotFoundError("Could not load mini_demo_data.json")import osprint("Data loading helper defined!")

In [ ]:
# Load the demo dataprint("Loading demo data...")data = load_data()# Extract documents from the loaded dataprint(f"Data loaded. Total examples: {data['metadata']['total_examples']}")print(f"Datasets: {data['metadata']['source_datasets']}")# Display first few examplesfor dataset in data['datasets']:    print(f"\nDataset: {dataset['dataset']}")    print(f"Number of examples: {len(dataset['examples'])}")

## Configuration

Set ALL tunable parameters to minimum values for fast demo execution. 

**Original parameters:** n_pairs=500, n_runs=50, k_values=[32, 64, 128]

**Demo parameters:** n_pairs=5, n_runs=3, k_values=[32]

In [ ]:
# Configuration - ALL tunable parameters in one place# Set to ABSOLUTE MINIMUM values for fast demoN_PAIRS = 5  # Number of document pairs to evaluate (Original: 500)N_RUNS = 3  # Number of runs per pair (Original: 50)K_VALUES = [32]  # k values for MinHash (Original: [32, 64, 128])N_BOOTSTRAP = 10  # Bootstrap samples (Original: 1000)CONFIDENCE = 0.95  # Confidence level for intervalsprint("Configuration set:")print(f"  N_PAIRS = {N_PAIRS}")print(f"  N_RUNS = {N_RUNS}")print(f"  K_VALUES = {K_VALUES}")print(f"  N_BOOTSTRAP = {N_BOOTSTRAP}")

## MinHash Implementation

The `MinHash` class computes MinHash signatures for documents using k hash functions.

**Key methods:**
- `get_shingles(text, k=3)`: Generate k-shingles from text
- `compute_signature(text)`: Compute MinHash signature

In [ ]:
# MinHash implementation (from original method.py)class MinHash:    """MinHash implementation with k hash functions."""    def __init__(self, k=128, seed=42):        self.k = k        self.seed = seed        self.seeds = [seed + i for i in range(k)]    def get_shingles(self, text, k=3):        """Generate k-shingles from text."""        words = text.lower().split()        if len(words) < k:            return set()        shingles = set()        for i in range(len(words) - k + 1):            shingle = ' '.join(words[i:i+k])            shingles.add(shingle)        return shingles    def compute_signature(self, text):        """Compute MinHash signature for a document."""        shingles = self.get_shingles(text)        if not shingles:            return [float('inf')] * self.k        signature = []        for seed in self.seeds:            min_hash = float('inf')            for shingle in shingles:                h = hashlib.md5(f"{seed}_{shingle}".encode()).hexdigest()                h_int = int(h[:8], 16)                h_normalized = h_int / (2**32)                min_hash = min(min_hash, h_normalized)            signature.append(min_hash)        return signature# Test MinHash with a sample documenttest_doc = data['datasets'][0]['examples'][0]['input']mh = MinHash(k=32, seed=42)sig = mh.compute_signature(test_doc)print(f"Test document: {test_doc[:60]}...")print(f"MinHash signature (k=32): {len(sig)} values")print(f"First 5 signature values: {sig[:5]}")

## Helper Functions

- `true_jaccard(set_a, set_b)`: Compute true Jaccard similarity
- `get_shingle_set(text, k=3)`: Get shingle set for true Jaccard
- `load_documents(data)`: Load documents from demo data

In [ ]:
def true_jaccard(set_a, set_b):    """Compute true Jaccard similarity between two sets."""    if not set_a or not set_b:        return 0.0    intersection = len(set_a & set_b)    union = len(set_a | set_b)    return intersection / union if union > 0 else 0.0def get_shingle_set(text, k=3):    """Get shingle set for true Jaccard computation."""    words = text.lower().split()    if len(words) < k:        return set()    shingles = set()    for i in range(len(words) - k + 1):        shingles.add(' '.join(words[i:i+k]))    return shinglesdef load_documents(data):    """Load and parse documents from the demo data structure."""    documents = []    for dataset in data['datasets']:        for example in dataset['examples']:            doc = {                'doc_id': example['metadata_doc_id'],                'text': example['input'],                'source': example['metadata_source'],                'word_count': example['metadata_word_count'],                'shingle_count': example['metadata_shingle_count']            }            documents.append(doc)    return documents# Test the helper functionsprint("Loading documents from demo data...")documents = load_documents(data)print(f"Loaded {len(documents)} documents")# Show true Jaccard between first two documentsdoc1_shingles = get_shingle_set(documents[0]['text'])doc2_shingles = get_shingle_set(documents[1]['text'])jaccard = true_jaccard(doc1_shingles, doc2_shingles)print(f"\nTrue Jaccard between doc 0 and doc 1: {jaccard:.4f}")

## Binomial Confidence Intervals

Implements three binomial CI methods:
1. **Clopper-Pearson**: Exact CI using Beta distribution
2. **Wilson score**: Wilson score interval
3. **Jeffreys**: Bayesian interval with Jeffreys prior

In [ ]:
class BinomialCI:    """Binomial confidence intervals for MinHash matching counts."""    @staticmethod    def clopper_pearson(x, n, confidence=0.95):        """Clopper-Pearson exact confidence interval."""        alpha = 1 - confidence        if x == 0:            lower = 0.0            upper = 1 - (alpha/2)**(1/n)        elif x == n:            lower = (alpha/2)**(1/n)            upper = 1.0        else:            lower = beta.ppf(alpha/2, x, n - x + 1)            upper = beta.ppf(1 - alpha/2, x + 1, n - x)        return (lower, upper)    @staticmethod    def wilson_score(x, n, confidence=0.95):        """Wilson score interval for binomial proportion."""        p_hat = x / n        z = stats.norm.ppf((1 + confidence) / 2)        denominator = 1 + z**2/n        center = (p_hat + z**2/(2*n)) / denominator        margin = z * np.sqrt(p_hat*(1-p_hat)/n + z**2/(4*n**2)) / denominator        lower = max(0, center - margin)        upper = min(1, center + margin)        return (lower, upper)    @staticmethod    def jeffreys(x, n, confidence=0.95):        """Jeffreys interval (Bayesian with Jeffreys prior)."""        alpha = 1 - confidence        posterior = beta(x + 0.5, n - x + 0.5)        lower = posterior.ppf(alpha/2)        upper = posterior.ppf(1 - alpha/2)        return (lower, upper)# Test Binomial CI methodsprint("Testing Binomial Confidence Interval methods...")print("\nFor x=5 matches out of n=32 hashes:")x, n = 5, 32print(f"\nClopper-Pearson: {BinomialCI.clopper_pearson(x, n)}")print(f"Wilson score: {BinomialCI.wilson_score(x, n)}")print(f"Jeffreys: {BinomialCI.jeffreys(x, n)}")

## Bayesian MinHash

Bayesian approach to MinHash uncertainty quantification using Beta prior informed by document length.

In [ ]:
class BayesianMinHash:    """Bayesian approach to MinHash uncertainty quantification."""    @staticmethod    def compute_prior_params(doc_length, method='uniform'):        """Compute Beta prior parameters based on document characteristics."""        if method == 'uniform':            return (1.0, 1.0)        elif method == 'jeffreys':            return (0.5, 0.5)        elif method == 'length_informed':            shingle_count = doc_length            if shingle_count < 50:                return (0.7, 0.7)            elif shingle_count < 100:                return (1.0, 1.0)            else:                return (2.0, 2.0)        else:            return (1.0, 1.0)    @staticmethod    def compute_posterior(x, n, alpha_prior, beta_prior):        """Compute posterior distribution given binomial likelihood."""        alpha_post = alpha_prior + x        beta_post = beta_prior + n - x        return beta(alpha_post, beta_post)    @staticmethod    def credible_interval(x, n, alpha_prior, beta_prior, confidence=0.95):        """Compute credible interval from posterior."""        posterior = BayesianMinHash.compute_posterior(x, n, alpha_prior, beta_prior)        alpha = 1 - confidence        lower = posterior.ppf(alpha/2)        upper = posterior.ppf(1 - alpha/2)        return (lower, upper)# Test Bayesian methodsprint("Testing Bayesian MinHash methods...")x, n = 5, 32# Uniform priorlower, upper = BayesianMinHash.credible_interval(x, n, 1.0, 1.0)print(f"\nUniform prior Beta(1,1):")print(f"  Credible interval: ({lower:.4f}, {upper:.4f})")

## Bootstrap MinHash

Bootstrap confidence intervals for MinHash (baseline method).

In [ ]:
class BootstrapMinHash:    """Bootstrap confidence intervals for MinHash."""    @staticmethod    def bootstrap_ci(sig_a, sig_b, k, n_bootstrap=1000, confidence=0.95):        """Compute bootstrap CI for Jaccard estimate."""        observed_matches = sum(1 for i in range(k) if abs(sig_a[i] - sig_b[i]) < 1e-10)        bootstrap_jaccards = []        for _ in range(n_bootstrap):            indices = np.random.choice(k, k, replace=True)            matches = sum(1 for i in indices if abs(sig_a[i] - sig_b[i]) < 1e-10)            bootstrap_jaccards.append(matches / k)        alpha = 1 - confidence        lower = np.percentile(bootstrap_jaccards, 100 * alpha/2)        upper = np.percentile(bootstrap_jaccards, 100 * (1 - alpha/2))        return (lower, upper)# Test Bootstrap methodprint("Testing Bootstrap MinHash method...")mh = MinHash(k=32, seed=0)doc1 = documents[0]['text']doc2 = documents[1]['text']sig1 = mh.compute_signature(doc1)sig2 = mh.compute_signature(doc2)matches = sum(1 for a, b in zip(sig1, sig2) if abs(a - b) < 1e-10)print(f"Observed matches: {matches} out of 32")lower, upper = BootstrapMinHash.bootstrap_ci(sig1, sig2, 32, n_bootstrap=N_BOOTSTRAP)print(f"Bootstrap CI: ({lower:.4f}, {upper:.4f})")

## Generate Document Pairs

Generate document pairs with known Jaccard similarity for evaluation.

In [ ]:
def generate_document_pairs(documents, n_pairs=1000, min_jaccard=0.0, max_jaccard=1.0, seed=42):    """Generate document pairs with known Jaccard similarity."""    pairs = []    np.random.seed(seed)    attempts = 0    while len(pairs) < n_pairs and attempts < n_pairs * 10:        i, j = np.random.choice(len(documents), 2, replace=False)        doc_i = documents[i]        doc_j = documents[j]        shingles_i = get_shingle_set(doc_i['text'])        shingles_j = get_shingle_set(doc_j['text'])        jaccard = true_jaccard(shingles_i, shingles_j)        if min_jaccard <= jaccard <= max_jaccard:            pairs.append({'doc_i': doc_i, 'doc_j': doc_j, 'true_jaccard': jaccard, 'index_i': i, 'index_j': j})        attempts += 1    return pairsprint(f"Generating {N_PAIRS} document pairs...")pairs = generate_document_pairs(documents, n_pairs=N_PAIRS, seed=42)print(f"Generated {len(pairs)} pairs")jaccards = [p['true_jaccard'] for p in pairs]print(f"True Jaccard range: [{min(jaccards):.4f}, {max(jaccards):.4f}]")

## Evaluate Coverage

Evaluate coverage probability of all CI methods.

In [ ]:
def evaluate_coverage(pairs, k_values=K_VALUES, n_runs=N_RUNS, n_bootstrap=N_BOOTSTRAP):    """Evaluate coverage probability of CI methods."""    results = {        'binomial_clopper_pearson': [],        'binomial_wilson': [],        'bayesian_uniform': [],        'bayesian_length_informed': [],        'bootstrap': []    }    for pair_idx, pair in enumerate(pairs):        doc_i = pair['doc_i']        doc_j = pair['doc_j']        true_J = pair['true_jaccard']        for k in k_values:            match_counts = []            for run in range(n_runs):                mh = MinHash(k=k, seed=run)                sig_i = mh.compute_signature(doc_i['text'])                sig_j = mh.compute_signature(doc_j['text'])                matches = sum(1 for a, b in zip(sig_i, sig_j) if abs(a - b) < 1e-10)                match_counts.append(matches)            for method_name in results.keys():                coverages = []                widths = []                for x in match_counts:                    if method_name == 'binomial_clopper_pearson':                        lower, upper = BinomialCI.clopper_pearson(x, k)                    elif method_name == 'binomial_wilson':                        lower, upper = BinomialCI.wilson_score(x, k)                    elif method_name == 'bayesian_uniform':                        lower, upper = BayesianMinHash.credible_interval(x, k, 1.0, 1.0)                    elif method_name == 'bayesian_length_informed':                        avg_shingles = (doc_i['shingle_count'] + doc_j['shingle_count']) / 2                        a, b = BayesianMinHash.compute_prior_params(avg_shingles, 'length_informed')                        lower, upper = BayesianMinHash.credible_interval(x, k, a, b)                    elif method_name == 'bootstrap':                        mh = MinHash(k=k, seed=0)                        sig_i = mh.compute_signature(doc_i['text'])                        sig_j = mh.compute_signature(doc_j['text'])                        lower, upper = BootstrapMinHash.bootstrap_ci(sig_i, sig_j, k, n_bootstrap=n_bootstrap)                    covered = (lower <= true_J <= upper)                    coverages.append(covered)                    widths.append(upper - lower)                results[method_name].append({                    'pair_idx': pair_idx, 'k': k, 'true_jaccard': true_J,                    'coverage': np.mean(coverages), 'avg_width': np.mean(widths),                    'match_counts': match_counts                })    return resultsprint("Evaluating coverage for all methods...")start_time = time.time()results = evaluate_coverage(pairs)elapsed = time.time() - start_timeprint(f"Evaluation complete in {elapsed:.2f} seconds")

## Benchmark Computation Time

Compare computational cost of each method.

In [ ]:
def benchmark_computation_time():    """Benchmark computation time for each method."""    k = 32    n_runs = 10    doc1 = "This is a test document with some words for benchmarking."    doc2 = "This is another test document with different words."    mh = MinHash(k=k, seed=0)    sig1 = mh.compute_signature(doc1)    sig2 = mh.compute_signature(doc2)    match_count = sum(1 for a, b in zip(sig1, sig2) if abs(a - b) < 1e-10)    results = {}    start = time.time()    for _ in range(n_runs):        BinomialCI.clopper_pearson(match_count, k)    results['clopper_pearson'] = (time.time() - start) / n_runs    start = time.time()    for _ in range(n_runs):        BinomialCI.wilson_score(match_count, k)    results['wilson'] = (time.time() - start) / n_runs    start = time.time()    for _ in range(n_runs):        BayesianMinHash.credible_interval(match_count, k, 1.0, 1.0)    results['bayesian'] = (time.time() - start) / n_runs    start = time.time()    for _ in range(n_runs):        BootstrapMinHash.bootstrap_ci(sig1, sig2, k, n_bootstrap=10)    results['bootstrap_10'] = (time.time() - start) / n_runs    return resultsprint("Benchmarking computation time...")timing_results = benchmark_computation_time()print("\nComputation time per method (microseconds):")for method, t in timing_results.items():    print(f"  {method}: {t * 1e6:.2f} us")

## Analyze and Visualize Results

Summarize the experiment results and visualize key findings.

In [ ]:
def analyze_results(results, timing):    """Analyze and summarize results."""    analysis = {'coverage_summary': {}, 'width_summary': {}, 'timing_summary': timing}    for method, result_list in results.items():        coverage_by_k = defaultdict(list)        width_by_k = defaultdict(list)        for r in result_list:            coverage_by_k[r['k']].append(r['coverage'])            width_by_k[r['k']].append(r['avg_width'])        analysis['coverage_summary'][method] = {k: {'mean': np.mean(v), 'std': np.std(v), 'target': 0.95} for k, v in coverage_by_k.items()}        analysis['width_summary'][method] = {k: {'mean': np.mean(v), 'std': np.std(v)} for k, v in width_by_k.items()}    return analysisprint("Analyzing results...")analysis = analyze_results(results, timing_results)print("\n" + "="*70)print("COVERAGE SUMMARY")print("="*70)for method, k_data in analysis['coverage_summary'].items():    for k, stats in k_data.items():        print(f"{method:<35} {stats['mean']:<15.4f} {stats['target']:<10.2f}")

In [ ]:
# Visualization: Coverage and Interval Width Comparisonimport matplotlib.pyplot as pltimport numpy as np# Prepare data for visualizationmethods = list(analysis['coverage_summary'].keys())coverage_means = []width_means = []for method in methods:    k = list(analysis['coverage_summary'][method].keys())[0]    coverage_means.append(analysis['coverage_summary'][method][k]['mean'])    width_means.append(analysis['width_summary'][method][k]['mean'])# Create subplotsfig, axes = plt.subplots(1, 3, figsize=(15, 5))# Plot 1: Coverage Probabilityax1 = axes[0]x_pos = np.arange(len(methods))ax1.bar(x_pos, coverage_means, align='center', alpha=0.7, color='steelblue')ax1.axhline(y=0.95, color='r', linestyle='--', label='Target (95%)')ax1.set_xlabel('Method')ax1.set_ylabel('Coverage Probability')ax1.set_title('Coverage Probability by Method')ax1.set_xticks(x_pos)xticklabels = [m.replace('binomial_', '').replace('bayesian_', '').replace('_', '\n') for m in methods]ax1.set_xticklabels(xticklabels, rotation=45, ha='right')ax1.legend()ax1.grid(True, alpha=0.3)# Plot 2: Interval Widthax2 = axes[1]ax2.bar(x_pos, width_means, align='center', alpha=0.7, color='coral')ax2.set_xlabel('Method')ax2.set_ylabel('Mean Interval Width')ax2.set_title('Interval Width by Method')ax2.set_xticks(x_pos)ax2.set_xticklabels(xticklabels, rotation=45, ha='right')ax2.grid(True, alpha=0.3)# Plot 3: Computation Timeax3 = axes[2]timing_methods = list(timing_results.keys())timing_values = [timing_results[m] * 1e6 for m in timing_methods]x_pos_t = np.arange(len(timing_methods))ax3.bar(x_pos_t, timing_values, align='center', alpha=0.7, color='mediumseagreen')ax3.set_xlabel('Method')ax3.set_ylabel('Time per Call (us)')ax3.set_title('Computation Time per Method')ax3.set_xticks(x_pos_t)ax3.set_xticklabels(timing_methods, rotation=45, ha='right')ax3.grid(True, alpha=0.3)plt.tight_layout()plt.show()